# Phase 2 — Orchestrator-Worker + Reflection

**Goal**: One question, answered by a *team* of agents instead of a single agent. An **Orchestrator** delegates the question to a **DataAgent** (the worker that queries data), then a **CriticAgent** reviews the worker's answer and either approves it or sends it back for a fix. Every delegation is recorded in the `Trace`.

In Phase 1 you had one agent + one tool. Here you learn the pattern that scales: split the work across specialized agents, and add a reviewer so mistakes get caught *before* the user sees them.

## Key concepts you'll learn

| Concept | Plain-English |
|---|---|
| **Worker** | A specialized agent — its own system prompt, its own tools. The `DataAgent` only knows how to query retail data. |
| **Orchestrator** | The coordinator. It doesn't answer the question itself — it *delegates* to workers and collects their results. Like a team lead. |
| **Delegation** | The orchestrator handing a task to a worker. Each delegation = one worker run. Counted in `trace.n_delegations`. |
| **Reflection / Critic** | A worker whose only job is to *review* another agent's output and return a verdict (APPROVE / REJECT). This is how an agent system catches its own mistakes. |
| **Revision loop** | If the critic rejects, the orchestrator re-delegates to the worker *with the critic's feedback* so it can fix the answer. |

## Acceptance criteria
1. The orchestrator delegates to **at least 2 workers** (DataAgent + CriticAgent) — `trace.n_delegations >= 2`
2. The CriticAgent produces a real verdict (APPROVE / REJECT)
3. The final answer matches the pandas ground truth
4. `Trace` exported to `traces/traces.jsonl` with `n_delegations` recorded

## 1. Setup

In [1]:
# Make the orchestrator/ library importable from notebooks/
import sys, os
from pathlib import Path

# Walk up to the orchestrator root (the folder containing 'data/retail.parquet').
# Idempotent: safe to re-run this cell any number of times.
ROOT = Path.cwd().resolve()
while not (ROOT / "data" / "retail.parquet").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
assert (ROOT / "data" / "retail.parquet").exists(), f"Could not find orchestrator root from {Path.cwd()}"
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv()

assert os.getenv("CLAUDE_CODE_OAUTH_TOKEN") or os.getenv("ANTHROPIC_API_KEY"), \
    "No auth token found in .env. Run `claude setup-token` and paste into .env."
print(f"[OK] Working dir: {os.getcwd()}")
print(f"[OK] Dev model: {os.getenv('CLAUDE_DEV_MODEL')}")

[OK] Working dir: /Users/ilynn/Desktop/Personal/2026/AI_Project/Multi_Agents Project/orchestrator
[OK] Dev model: claude-haiku-4-5-20251001


In [2]:
import pandas as pd

from orchestrator.tools import get_retail_df
from orchestrator import tools
from orchestrator.observability import Trace, Timer
# The Phase 2 machinery you'll use to run and review workers:
from orchestrator.agents import AgentRun, run_agent, parse_verdict
from claude_agent_sdk import tool, create_sdk_mcp_server, ClaudeAgentOptions

DEV_MODEL = os.getenv("CLAUDE_DEV_MODEL", "claude-haiku-4-5-20251001")

df = get_retail_df()
print(f"Loaded {len(df):,} rows")
print(f"Dev model: {DEV_MODEL}")

Loaded 1,067,371 rows
Dev model: claude-haiku-4-5-20251001


## 2. Ground truth

Same evals-first mindset as Phase 1: compute the correct answer with pandas *before* trusting the agents. This phase's question is about **countries** (so you exercise a different `group_by` than Phase 1's SKUs).

In [3]:
# Top 5 countries by revenue in 2010 — computed directly with pandas
df_2010 = df[(df["Quantity"] > 0) & (df["InvoiceDate"].dt.year == 2010)]

ground_truth = (
    df_2010.groupby("Country")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
)
ground_truth

,Country,revenue
0,United Kingdom,8707529.003
1,EIRE,370911.340
2,Netherlands,262365.750
3,Germany,207539.611
4,France,150306.110


In [17]:
df["InvoiceDate"].agg(["min", "max"])


min   2009-12-01 07:45:00
max   2011-12-09 12:50:00
Name: InvoiceDate, dtype: datetime64[us]

## 3. Rebuild the retail tool + MCP server (recap from Phase 1)

The `DataAgent` worker needs the `query_retail` tool. This cell is the *working* version from the end of Phase 1 — note `country` is **optional** in the schema (the fix you made when the agent kept refusing to answer). Nothing to change here; just run it.

In [4]:
@tool(
    "query_retail",
    description=(
        "Query the retail transactions dataset to return the top N entries "
        "ranked by a metric. Use this for any 'top N' question about products, "
        "countries, or customers. Arguments: year (optional), country (optional "
        "- OMIT to include all countries), top_n (default 10), group_by (one of "
        "'StockCode', 'Country', 'Customer ID'), metric (one of 'revenue', "
        "'Quantity'). Returns a list of dicts, one row each."
    ),
    input_schema={
        "type": "object",
        "properties": {
            "year":     {"type": "integer", "description": "Calendar year filter, e.g. 2010"},
            "country":  {"type": "string",  "description": "Optional country filter. OMIT to include all countries."},
            "top_n":    {"type": "integer", "description": "How many rows to return"},
            "group_by": {"type": "string",  "description": "'StockCode', 'Country', or 'Customer ID'"},
            "metric":   {"type": "string",  "description": "'revenue' or 'Quantity'"},
        },
        "required": [],
    },
)
async def query_retail_tool(args):
    """Adapter: the agent passes args as a dict; we call the pandas function."""
    rows = tools.query_retail(
        year=args.get("year"),
        country=args.get("country"),
        top_n=args.get("top_n", 10),
        group_by=args.get("group_by", "StockCode"),
        metric=args.get("metric", "revenue"),
    )
    import json
    return {"content": [{"type": "text", "text": json.dumps(rows, default=str)}]}


retail_server = create_sdk_mcp_server(name="retail", version="1.0.0", tools=[query_retail_tool])
print("[OK] Retail tool + MCP server ready")

[OK] Retail tool + MCP server ready


## 4. Worker #1 — the DataAgent

A **worker** is just an agent with a focused job. The `DataAgent` does one thing: take a question, query the retail data, return a tabled answer. It's configured exactly like the Phase 1 agent — a `ClaudeAgentOptions` with the retail tool.

The difference from Phase 1 is *who calls it*. In Phase 1 you called the agent directly. Here the **orchestrator** will call it, via the `run_agent(...)` helper from `orchestrator/agents.py`.

In [5]:
data_options = ClaudeAgentOptions(
    model=DEV_MODEL,
    system_prompt=(
        "You are a retail data analyst. Answer questions about retail "
        "transactions using the query_retail tool. Call the tool with the "
        "correct filters, then present the result as a clean markdown table. "
        "Only filter by country if the user explicitly names a country. "
        "Never invent numbers - report exactly what the tool returns. "
        "If a reviewer gives you feedback on a previous answer, read it "
        "carefully and correct the specific problem they identified."
    ),
    mcp_servers={"retail": retail_server},
    allowed_tools=["mcp__retail__query_retail"],
    max_turns=5,
)
print("[OK] DataAgent configured")

[OK] DataAgent configured


## 5. Worker #2 — the CriticAgent (reflection)

The **CriticAgent** never touches the data. Its job is to *review* the DataAgent's answer and decide: is this good enough to show the user?

This is **reflection** — an agent system checking its own work. The critic catches *semantic* mistakes: an implausible number, a missing part of the question, an answer that contradicts what we know about the data.

For the orchestrator to act on the verdict, the critic's reply must be **machine-readable**. The `parse_verdict()` helper (in `agents.py`) reads the critic's first word — so your prompt must make the critic start with **APPROVE** or **REJECT**.

**TODO 1**: write the CriticAgent's system prompt.

In [6]:
critic_system_prompt = (
    # ----- TODO 1: write the CriticAgent's system prompt -----------------
    # The critic REVIEWS a proposed answer. It does NOT query data - it
    # reasons about whether the answer looks correct and complete.
    #
    # Your prompt MUST tell the critic to:
    #   - reply with the single word APPROVE or REJECT as the FIRST word
    #   - if REJECT, follow it with a one-sentence reason
    #   - check: does the answer actually address the question? are any
    #     numbers obviously implausible? is anything required missing?
    #
    # Write 3-5 sentences. Imagine briefing a picky QA reviewer on day one.
    """"You are a critique agent. Your sole purpose is to review the DataAgent's 
answer to a question. Start your reply with a single bare word — APPROVE
or REJECT — as the very first word, with no quotes or punctuation before
it. If you REJECT, follow it with one sentence explaining what is wrong.
Check: does the answer actually address the question, are any numbers
implausible, and is anything required missing? Note that this is
UK-dominated retail data, so a top-countries answer that omits the United
Kingdom is almost certainly wrong."""
    # ---------------------------------------------------------------------
    
)

critic_options = ClaudeAgentOptions(
    model=DEV_MODEL,
    system_prompt=critic_system_prompt,
    # NOTE: the critic has NO mcp_servers and NO tools - it only reasons
    # about the text it is given. That is the whole point of a reviewer.
    max_turns=2,
)
print("[OK] CriticAgent configured")

[OK] CriticAgent configured


## 6. Watch the Critic catch a bad answer

Before wiring everything together, prove the critic actually *works*. We hand it a deliberately **wrong** answer (it claims France led 2010 revenue — but you know from the ground truth that the UK dominates this dataset).

If your TODO 1 prompt is good, the critic should reply starting with **REJECT**.

In [19]:
demo_question = "What were the top 5 countries by revenue in 2010?"
fake_bad_answer = (
    "The top countries by revenue in 2010 were: 1. France, 2. Spain, "
    "3. Italy, 4. Portugal, 5. Greece. The United Kingdom was not in the top 5."
)

demo_prompt = f"QUESTION:\n{demo_question}\n\nPROPOSED ANSWER:\n{fake_bad_answer}"
demo_run = await run_agent("CriticAgent", demo_prompt, critic_options)

print("----- Critic verdict on the BAD answer -----")
print(demo_run.answer)

approved, reason = parse_verdict(demo_run.answer)
print(f"\nparse_verdict() -> approved={approved}")
assert not approved, "The critic APPROVED a clearly wrong answer - strengthen your TODO 1 prompt."
print("[OK] Critic correctly rejected the bad answer")

----- Critic verdict on the BAD answer -----
REJECT The answer omits the United Kingdom from the top 5 despite the system note that this is UK-dominated retail data, making it almost certainly incorrect for a UK retail dataset.

parse_verdict() -> approved=False
[OK] Critic correctly rejected the bad answer


## 7. The Orchestrator — delegate, reflect, revise

Now the coordinator. The `orchestrate()` function ties the workers together:

1. **Delegate** the question to the `DataAgent` -> get a first answer
2. **Delegate** that answer to the `CriticAgent` -> get a verdict
3. If **APPROVE** -> done
4. If **REJECT** -> re-delegate to the `DataAgent` with the critic's feedback, then loop back to step 2

Every worker call is one **delegation**. The orchestrator itself writes no answer — it only routes.

**TODO 2**: fill in the revision step — when the critic rejects, send the work back to the DataAgent.

In [ ]:
async def orchestrate(question, data_options, critic_options, max_revisions=1):
    """Orchestrator: DataAgent answers, CriticAgent reviews, DataAgent revises if rejected."""
    runs = []            # every AgentRun, for token/cost accounting
    n_delegations = 0

    # --- Delegation 1: DataAgent produces a first answer ---
    data_run = await run_agent("DataAgent", question, data_options)
    runs.append(data_run)
    n_delegations += 1
    answer = data_run.answer
    print(f"[delegate] DataAgent  -> produced {len(answer)} chars")

    # --- Reflection loop: Critic reviews; DataAgent revises if rejected ---
    approved = False
    for attempt in range(max_revisions + 1):
        critic_prompt = f"QUESTION:\n{question}\n\nPROPOSED ANSWER:\n{answer}"
        critic_run = await run_agent("CriticAgent", critic_prompt, critic_options)
        runs.append(critic_run)
        n_delegations += 1

        approved, reason = parse_verdict(critic_run.answer)
        print(f"[review]   CriticAgent -> {'APPROVE' if approved else 'REJECT'}")

        if approved or attempt == max_revisions:
            break

        # ----- TODO 2: the critic REJECTED - re-delegate to the DataAgent -----
        # Build `revision_prompt` containing BOTH the original question AND
        # the critic's feedback (`reason`), so the DataAgent can fix the
        # mistake. Then run the DataAgent again, record the run, count the
        # delegation, and update `answer`.
        #
        # Fill in the 4 lines below (replace each `...`):
        revision_prompt = (
            f"{question}\n\n"
            f"A reviewer rejected your previous answer. their feedback: \n"
            f"{reason}\n\n"
            f"read the feadback, understand what went wrong, and give a corrected answer."
        )
        data_run = await run_agent('DataAgent', revision_prompt, data_options)                      # await run_agent("DataAgent", revision_prompt, data_options)
        runs.append(data_run)                     # the new data_run
        n_delegations += 1
        answer = data_run.answer                         # data_run.answer
        print(f"[delegate] DataAgent  -> revised answer ({len(answer)} chars)")
        # ----------------------------------------------------------------------

    return {"answer": answer, "approved": approved,
            "runs": runs, "n_delegations": n_delegations}

print("[OK] Orchestrator defined")

[OK] Orchestrator defined


## 8. Run the orchestrator

**TODO 3**: write the business question. Ask for the **top 5 countries by revenue in 2010** — phrase it naturally, like you would to a colleague.

In [18]:
# ----- TODO 3: write your business question -----
QUESTION = "what are the top 5 countries that drive the most revenue in 2010 "
# ------------------------------------------------

trace = Trace(model=DEV_MODEL, question=QUESTION)

with Timer() as t:
    result = await orchestrate(QUESTION, data_options, critic_options, max_revisions=1)

# Roll every worker's usage into the single Trace for this orchestrated run
for r in result["runs"]:
    trace.input_tokens        += r.input_tokens
    trace.output_tokens       += r.output_tokens
    trace.cached_input_tokens += r.cached_input_tokens
    trace.n_tool_calls        += r.n_tool_calls
trace.n_delegations = result["n_delegations"]
trace.latency_ms    = t.elapsed_ms
trace.answer        = result["answer"]
trace.compute_cost()

print("\n----- Final answer -----")
print(result["answer"])
print(f"\nDelegations: {trace.n_delegations} | Tool calls: {trace.n_tool_calls} "
      f"| Approved: {result['approved']}")

[delegate] DataAgent  -> produced 522 chars
[review]   CriticAgent -> APPROVE

----- Final answer -----
Here are the **top 5 countries by revenue in 2010**:

| Rank | Country | Revenue |
|------|---------|---------|
| 1 | United Kingdom | $8,707,529.00 |
| 2 | EIRE | $370,911.34 |
| 3 | Netherlands | $262,365.75 |
| 4 | Germany | $207,539.61 |
| 5 | France | $150,306.11 |

The **United Kingdom** is by far the dominant market, generating over $8.7 million in revenue—significantly more than all other countries combined. EIRE (Ireland) is the second-largest market but with considerably less revenue at approximately $371k.

Delegations: 2 | Tool calls: 1 | Approved: True


## 9. Verify (acceptance criteria)

In [12]:
ground_truth


,Country,revenue
0,United Kingdom,8707529.003
1,EIRE,370911.340
2,Netherlands,262365.750
3,Germany,207539.611
4,France,150306.110


In [ ]:
# ----- TODO 4: write the assertion -----
# Check that every country from the ground truth appears in the final answer.
missing_countries = []
for country in ground_truth["Country"].tolist():
    if country not in result["answer"]:
        missing_countries.append(country)
# ---------------------------------------

trace.passed = (missing_countries == [])
print(f"Missing countries: {missing_countries}")
print(f"Passed: {trace.passed}")
assert trace.passed, f"Orchestrator missed: {missing_countries}"

# Phase 2 acceptance criteria
assert trace.n_delegations >= 2, \
    f"Expected >=2 delegations (DataAgent + CriticAgent), got {trace.n_delegations}"
assert result["approved"], "Critic did not approve the final answer"

trace.append_jsonl()
print("\n[OK] Acceptance criteria met - delegations recorded, critic approved, trace saved")

## 10. Quiz (~5 min — answer in a new markdown cell)

**TODO 5**: Answer in your own words.

1. **Worker vs orchestrator**: what is the difference between the DataAgent and the orchestrator? Which one actually *answers* the question, and which one only *routes*?

   The **DataAgent** does the actual work — it has the `query_retail` tool, runs the SQL-equivalent, and writes the markdown table. The **orchestrator** has no tools and writes no answer itself; it's a pure Python coordinator that decides *which worker runs next* and *what prompt they get*. Think of the orchestrator as a tech lead at standup and the DataAgent as the engineer who actually opens the IDE.

2. **Why reflection**: the CriticAgent costs extra tokens and latency on every run. Give one concrete kind of mistake it can catch that the DataAgent alone would not — and explain why the DataAgent can't reliably catch its own mistake.

   The critic catches answers that are *internally consistent but contextually wrong* — e.g. the DataAgent dutifully reports "top 5 countries in 2010: France, Spain, Italy, Portugal, Greece" because it called the tool with the wrong filter, and the result *looks* fine on its own. The DataAgent can't catch this reliably because it's the same model that just produced the answer — it has the same blind spot it had a second ago. A fresh agent with a *reviewer* system prompt approaches the same text from a different angle (and can be primed with sanity checks like "UK-dominated dataset → UK should be #1").

3. **Delegations**: on a run where the critic APPROVES the first answer, what is `trace.n_delegations`? On a run where it REJECTS once then approves, what is it? Why?

   Approve-first-try → **2** delegations (DataAgent + CriticAgent). One reject + then approve → **4** (DataAgent, CriticAgent rejects, DataAgent revises, CriticAgent approves). Every worker invocation counts as one delegation; the orchestrator itself isn't a delegation.

4. **The verdict format**: why must the critic start its reply with the literal word APPROVE or REJECT? What would break if it just replied "Yeah this looks fine to me"?

   `parse_verdict()` reads only the *first word* and matches against `"APPROVE"`. "Yeah this looks fine to me" → first word is `YEAH` → parser sees `approved=False` → the orchestrator triggers an unnecessary revision loop, doubling cost and latency for a perfectly good answer. Free-text verdicts are a UX win for humans and a parsing nightmare for machines — pin the format so the contract is unambiguous.

5. **Cost trade-off**: this phase roughly doubles the number of LLM calls vs Phase 1. When is that worth it, and when would you skip the critic to save money?

   Worth it when (a) a wrong answer has real downstream cost — exec dashboards, finance numbers, anything someone will *act* on, and (b) the worker model is a small cheap one (Haiku) prone to subtle mistakes the critic can catch. Skip it when the task is structurally simple ("format these 10 rows as a table"), the worker is already a strong model, or you're in an interactive loop where the human is themselves the critic.

## Phase 2 done when:
- [x ] TODO 1 (CriticAgent system prompt) filled in
- [x ] TODO 2 (revision step in the orchestrator) filled in
- [x ] TODO 3 (your question) filled in
- [x ] TODO 4 (assertion) filled in
- [ x] TODO 5 (quiz) answered
- [ ]x Cell 13 shows the critic REJECTing the bad answer
- [ x] Notebook runs top-to-bottom without errors
- [x ] `traces/traces.jsonl` has a new line with `n_delegations >= 2`

Then ping me — we'll review your critic prompt + quiz answers, then move to Phase 3 (planning & decomposition).